# 🏷️ Support Vector Machines (SVM) — Solutions Notebook

**Complete, verified solutions.**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print('Setup complete! ✅')

### Linear SVM from Scratch Solution

In [ ]:
# ✅ SOLUTION: Linear SVM using Subgradient Descent on Hinge Loss
class LinearSVMFromScratch:
    def __init__(self, C=1.0, lr=0.001, num_iters=1000):
        self.C = C
        self.lr = lr
        self.num_iters = num_iters
        self.w = None
        self.b = None
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Ensure labels are in {-1, +1}
        y_cls = np.where(y <= 0, -1, 1)
        
        self.w = np.zeros(n_features)
        self.b = 0.0
        
        for iteration in range(self.num_iters):
            for idx, x_i in enumerate(X):
                condition = y_cls[idx] * (np.dot(x_i, self.w) + self.b) >= 1
                if condition:
                    self.w -= self.lr * (1 / n_samples) * self.w
                else:
                    self.w -= self.lr * ((1 / n_samples) * self.w - self.C * y_cls[idx] * x_i)
                    self.b -= self.lr * (-self.C * y_cls[idx])
                    
    def predict(self, X):
        approx = np.dot(X, self.w) + self.b
        return np.where(approx >= 0, 1, 0)

# Test implementation
X_blob, y_blob = make_blobs(n_samples=200, centers=2, n_features=2, random_state=42, cluster_std=1.2)
svm_custom = LinearSVMFromScratch(C=1.0, lr=0.001, num_iters=2000)
svm_custom.fit(X_blob, y_blob)
preds = svm_custom.predict(X_blob)

print(f'Linear SVM (From Scratch) Accuracy: {accuracy_score(y_blob, preds):.4f}')

### Nonlinear Decision Boundaries: Linear vs RBF Kernel

In [ ]:
X_moon, y_moon = make_moons(n_samples=300, noise=0.2, random_state=42)

svm_lin = SVC(kernel='linear', C=1.0)
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')

svm_lin.fit(X_moon, y_moon)
svm_rbf.fit(X_moon, y_moon)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Helper function to plot decision boundary
def plot_boundary(model, ax, title):
    x_min, x_max = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
    y_min, y_max = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap='bwr', edgecolors='k', alpha=0.8)
    ax.set_title(title)

plot_boundary(svm_lin, axes[0], f'Linear Kernel (Acc: {svm_lin.score(X_moon, y_moon):.2f})')
plot_boundary(svm_rbf, axes[1], f'RBF Kernel (Acc: {svm_rbf.score(X_moon, y_moon):.2f})')

plt.tight_layout()
plt.show()